# Conflict Detection & Visual Analysis

## Overview

In a multi-agent system, the most common coordination failures happen when one agent operates on stale information while another has already received an update. This notebook evaluates how a hub-and-spoke multi-agent architecture handles conflicting information by running two sessions:

1. **Baseline** — A straightforward trip request with a fixed budget and no mid-session changes.
2. **Conflict** — The user changes their budget and dates mid-session, testing whether all agents pick up the update and produce a consistent result.

Both sessions are scored using LLM-as-judge semantic metrics (Context Freshness, Handoff Completeness, Context Utilization, State Consistency, Memory Write Accuracy, and Redundant Context) alongside static latency and token metrics. Results are presented as visual dashboards for quick interpretation.

## Architecture

```
                    ┌─────────────────────┐
                    │   User / Customer    │
                    └──────────┬──────────┘
                               │
                    ┌──────────▼──────────┐
                    │   Travel Coordinator │  ← Hub
                    │   Routes queries to  │    Decides which spoke to call
                    │   specialized agents │    Compresses user message
                    └──┬───────┬────────┬──┘    into handoff query
                       │       │        │
              ┌────────▼──┐ ┌──▼─────┐ ┌▼──────────┐
              │  Flight   │ │ Hotel  │ │ Itinerary │
              │  Agent    │ │ Agent  │ │ Agent     │
              └─────┬─────┘ └───┬────┘ └─────┬─────┘
                    │           │             │
              ┌─────▼───────────▼─────────────▼─────┐
              │     Shared Memory (Python list)     │
              │     Each agent reads prior entries   │
              │     and appends its own response     │
              └─────────────────────────────────────┘
```

The coordinator receives user requests and delegates to three specialized spoke agents (Flight, Hotel, Itinerary) via tool calls. Each spoke reads from a shared in-process Python list before processing and appends its response after. The Itinerary agent runs last, consuming Flight and Hotel outputs from shared memory to produce a cohesive plan.

In the conflict session, the user changes constraints mid-conversation. The evaluation measures whether the coordinator propagates the update, whether downstream agents read the latest context, and whether the final itinerary reflects the new constraints.

In [ ]:
import os
import time
import logging
import copy

from IPython.display import display, Markdown
import matplotlib.pyplot as plt

from strands import Agent, tool
from strands.hooks import HookProvider, HookRegistry
from strands.hooks.events import AgentInitializedEvent, MessageAddedEvent

from model_config import (
    AGENT_MODEL_ID, FLIGHT_PROMPT, HOTEL_PROMPT, ITINERARY_PROMPT, HUB_PROMPT
)
from eval_helpers import (
    format_memory, print_memory,
    plot_context_metrics_radar, plot_latency_breakdown,
    plot_session_comparison, plot_coordination_overhead,
    compute_embeddings, plot_c2_heatmap
)
from metrics_collector import MetricsCollector

logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger("conflict-notebook")

region = os.environ.get("AWS_REGION", "us-west-2")
print(f"Region: {region}")
print(f"Agent model: {AGENT_MODEL_ID}")

## 1. Memory Hook

Each spoke agent uses a hook that reads shared memory on initialization and writes its response back after processing. This enables agents to build on each other's outputs.

In [ ]:
class ListMemoryHook(HookProvider):
    """Hook that reads/writes a shared Python list."""

    def __init__(self, memory: list, collector: MetricsCollector, agent_name: str):
        self.memory = memory
        self.collector = collector
        self.agent_name = agent_name

    def on_agent_initialized(self, event: AgentInitializedEvent):
        t0 = time.perf_counter()
        context = format_memory(self.memory)
        read_latency = time.perf_counter() - t0
        self.collector.record_memory_read_latency(self.agent_name, read_latency)
        if context:
            event.agent.system_prompt += (
                f"\n\nShared memory from other agents:\n{context}"
                "\n\nUse this context. Reference specific details from other agents.")
        self.collector.record_retrieved_context(self.agent_name, context)

    def on_message_added(self, event: MessageAddedEvent):
        last = event.agent.messages[-1]
        if last["role"] != "assistant":
            return
        text_parts = [b["text"] for b in last.get("content", []) if isinstance(b, dict) and b.get("text")]
        if not text_parts:
            return
        response_text = "\n".join(text_parts)
        self.collector.record_response(self.agent_name, response_text)
        t0 = time.perf_counter()
        self.memory.append({
            "agent": self.agent_name, "role": "assistant",
            "content": response_text, "ts": time.time()})
        self.collector.record_memory_write_latency(self.agent_name, time.perf_counter() - t0)

    def register_hooks(self, registry: HookRegistry):
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        registry.add_callback(MessageAddedEvent, self.on_message_added)

## 2. Session Runner

In [ ]:
def run_session(conversation: list, session_label: str) -> tuple:
    """Run a full conversation session. Returns (collector, shared_memory)."""
    shared_memory = []
    collector = MetricsCollector(region=region)
    turn_counter = 0

    @tool
    def flight_booking_assistant(query: str) -> str:
        """Process flight booking queries.
        Args:
            query: A flight-related question
        Returns:
            Flight information and booking options
        """
        collector.record_handoff("flight", query)
        hook = ListMemoryHook(shared_memory, collector, "flight")
        agent = Agent(hooks=[hook], model=AGENT_MODEL_ID, system_prompt=FLIGHT_PROMPT)
        t0 = time.perf_counter()
        resp = agent(query)
        collector.record_agent_latency("flight", time.perf_counter() - t0)
        usage = getattr(resp.metrics, "accumulated_usage", {}) if hasattr(resp, "metrics") else {}
        collector.record_token_usage("flight", usage.get("inputTokens", 0), usage.get("outputTokens", 0))
        return str(resp)

    @tool
    def hotel_booking_assistant(query: str) -> str:
        """Process hotel booking queries.
        Args:
            query: A hotel-related question
        Returns:
            Hotel information and booking options
        """
        collector.record_handoff("hotel", query)
        hook = ListMemoryHook(shared_memory, collector, "hotel")
        agent = Agent(hooks=[hook], model=AGENT_MODEL_ID, system_prompt=HOTEL_PROMPT)
        t0 = time.perf_counter()
        resp = agent(query)
        collector.record_agent_latency("hotel", time.perf_counter() - t0)
        usage = getattr(resp.metrics, "accumulated_usage", {}) if hasattr(resp, "metrics") else {}
        collector.record_token_usage("hotel", usage.get("inputTokens", 0), usage.get("outputTokens", 0))
        return str(resp)

    @tool
    def itinerary_assistant(query: str) -> str:
        """Build a travel itinerary from flight and hotel results.
        Args:
            query: Request to build an itinerary
        Returns:
            A cohesive travel itinerary
        """
        collector.record_handoff("itinerary", query)
        hook = ListMemoryHook(shared_memory, collector, "itinerary")
        agent = Agent(hooks=[hook], model=AGENT_MODEL_ID, system_prompt=ITINERARY_PROMPT)
        t0 = time.perf_counter()
        resp = agent(query)
        collector.record_agent_latency("itinerary", time.perf_counter() - t0)
        usage = getattr(resp.metrics, "accumulated_usage", {}) if hasattr(resp, "metrics") else {}
        collector.record_token_usage("itinerary", usage.get("inputTokens", 0), usage.get("outputTokens", 0))
        return str(resp)

    hub = Agent(system_prompt=HUB_PROMPT, model=AGENT_MODEL_ID,
               tools=[flight_booking_assistant, hotel_booking_assistant, itinerary_assistant])

    for msg in conversation:
        turn_counter += 1
        collector.begin_turn(turn_counter, msg)
        print(f"\n{'='*60}")
        print(f"[{session_label}] Turn {turn_counter}: {msg}")
        print(f"{'='*60}")
        resp = hub(msg)
        collector.end_turn()
        print(f"\nHub: {str(resp)[:300]}...")

    return collector, shared_memory

## 3. Session A — Baseline (no conflict)

Simple trip request with a fixed budget. No mid-session changes.

In [ ]:
baseline_conversation = [
    "Book a trip from LA to NYC, July 10 to July 17, 1 traveler. "
    "Budget $1800 for flights. I want morning flights and a hotel in Midtown with a pool.",
]

baseline_metrics, baseline_memory = run_session(baseline_conversation, "Baseline")

## 4. Session B — Conflicting Information

The user changes the budget mid-session. This tests whether:
- The coordinator propagates the update to all agents
- Agents that already ran pick up the new constraint
- The itinerary agent reconciles conflicting flight/hotel data

This is the most common real-world failure: **one agent works with stale context while another has the update.**

In [ ]:
conflict_conversation = [
    "Book a trip from LA to NYC, July 10 to July 17, 1 traveler. "
    "Budget $1800 for flights. I want morning flights and a hotel in Midtown with a pool.",
    
    "Actually, my budget just dropped to $900 total for flights AND hotel combined. "
    "I need the cheapest options. No pool requirement anymore. "
    "Also change dates to July 15-20.",
    
    "Now build the final itinerary with the updated budget and dates."
]

conflict_metrics, conflict_memory = run_session(conflict_conversation, "Conflict")

## 5. Evaluate Both Sessions

In [ ]:
print("Evaluating baseline session...")
baseline_metrics.evaluate_all()

print("Evaluating conflict session...")
conflict_metrics.evaluate_all()

print("Done.")

## 6. Context Flow Traces

Before looking at scores, inspect what each agent saw and produced.

In [ ]:
display(Markdown("## Baseline Session\n\n" + baseline_metrics.trace_report()))

In [ ]:
display(Markdown("## Conflict Session\n\n" + conflict_metrics.trace_report()))

## 7. Visual Analysis — Radar Charts

Each axis is a context quality metric (1-5). A perfect system fills the entire pentagon.

In [ ]:
fig = plot_context_metrics_radar(baseline_metrics, "Baseline (no conflict)")
plt.show()

fig = plot_context_metrics_radar(conflict_metrics, "Conflict (budget change)")
plt.show()

## 8. Visual Analysis — Latency Breakdown

How much time is coordination (memory read/write) vs actual reasoning?

In [ ]:
fig = plot_latency_breakdown(baseline_metrics, "Baseline")
plt.show()

fig = plot_latency_breakdown(conflict_metrics, "Conflict")
plt.show()

## 9. Visual Analysis — Token Usage

How many input and output tokens does each agent call consume? This helps identify which agents are doing the most work and where token costs concentrate.

In [ ]:
fig = plot_coordination_overhead(baseline_metrics, "Baseline")
plt.show()

fig = plot_coordination_overhead(conflict_metrics, "Conflict")
plt.show()

## 10. Side-by-Side Comparison

Direct comparison of context quality between the two sessions. The conflict session should show lower scores on **Context Freshness** and **State Consistency** — those are the metrics that detect stale/conflicting information.

In [ ]:
fig = plot_session_comparison(baseline_metrics, conflict_metrics,
                              "Baseline", "Conflict")
plt.show()

In [ ]:
display(Markdown(MetricsCollector.comparison_report(
    baseline_metrics, conflict_metrics,
    label_a="Baseline", label_b="Conflict")))

## 11. Detailed Metric Tables

In [ ]:
display(Markdown("## Baseline\n\n" + baseline_metrics.context_metrics_report()))
display(Markdown(baseline_metrics.latency_metrics_report()))

In [ ]:
display(Markdown("## Conflict\n\n" + conflict_metrics.context_metrics_report()))
display(Markdown(conflict_metrics.latency_metrics_report()))

## 12. C2 Alignment — Do Agents Converge?

Embedding similarity between agent responses. In the conflict session, we expect lower alignment between agents that saw different versions of the budget.

In [ ]:
# Build final responses from memory
baseline_final = {}
for e in baseline_memory:
    if e.get("role") == "assistant":
        baseline_final[e["agent"]] = e["content"]

conflict_final = {}
for e in conflict_memory:
    if e.get("role") == "assistant":
        conflict_final[e["agent"]] = e["content"]

print("Computing embeddings for baseline...")
baseline_emb = compute_embeddings(baseline_final, region=region)

print("\nComputing embeddings for conflict...")
conflict_emb = compute_embeddings(conflict_final, region=region)

In [ ]:
if baseline_emb:
    fig = plot_c2_heatmap(baseline_emb, "Baseline")
    plt.show()

if conflict_emb:
    fig = plot_c2_heatmap(conflict_emb, "Conflict")
    plt.show()

## 13. Key Takeaways

**What the conflict session reveals:**

- **Context Freshness** drops when an agent reads memory that doesn't reflect the latest user message (e.g., hotel agent still sees $1800 budget after user changed to $900)
- **State Consistency** drops when agents disagree on key facts (flight agent uses old dates, hotel agent uses new dates)
- **Handoff Completeness** may drop if the coordinator doesn't include the updated constraints in the handoff query
- **C2 Alignment** diverges when agents produce responses based on different versions of the same facts

**How to handle conflicts in production:**

1. **Version your memory entries** — tag each write with a monotonic version number so readers can detect stale reads
2. **Re-dispatch after updates** — when the user changes constraints, the coordinator should re-run affected agents, not just the next one
3. **Add a reconciliation step** — before the final agent (itinerary), run a consistency check that compares all agents' assumptions
4. **Monitor these metrics in production** — Context Freshness < 3 and State Consistency < 3 are signals that your coordination layer needs attention

## 14. Try It Yourself — Compare Models

The scores you see above depend heavily on which model powers the coordinator and spoke agents. A stronger model will propagate constraint changes more reliably, re-dispatch agents when needed, and produce more consistent results.

To compare models, open `model_config.py` and change `AGENT_MODEL_ID` to a different model. For example:

```python
# Try these alternatives:
AGENT_MODEL_ID = "us.amazon.nova-pro-v1:0"        # Amazon Nova Pro
AGENT_MODEL_ID = "us.amazon.nova-lite-v1:0"       # Amazon Nova Lite (faster, less capable)
AGENT_MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"  # Claude Sonnet (default)
```

Then restart the kernel and re-run the notebook from the top. Compare the results:

- Does the coordinator re-dispatch the flight agent after the budget change?
- Do the Context Freshness and State Consistency scores differ between models?
- How does the C2 alignment heatmap change — do agents converge more or less?
- Is there a meaningful difference in token usage or latency?

This is a practical way to evaluate which foundation model best fits your multi-agent architecture before committing to production.